In [4]:
# ============================================================
# TASK 2 - GOOGLE PLAY STORE ANALYTICS
# AUTOMATIC DISPLAY: 6 PM TO 8 PM IST
# ============================================================

import os
import webbrowser
import pandas as pd
import numpy as np
import plotly.express as px

# ============================================================
# 1. LOAD DATASET
# ============================================================

file_path = "Play Store Data.csv"
apps_df = pd.read_csv(file_path)

print("Dataset loaded:", apps_df.shape)

# ============================================================
# 2. CLEAN INSTALLS COLUMN
# ============================================================

task2_df = apps_df.copy()

task2_df["Installs"] = (
    task2_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

task2_df["Installs"] = pd.to_numeric(
    task2_df["Installs"], errors="coerce"
)

task2_df = task2_df.dropna(
    subset=["Installs", "Category"]
).copy()

task2_df["Installs"] = task2_df["Installs"].astype(int)

# ============================================================
# 3. EXCLUDE CATEGORIES STARTING WITH A, C, G AND S
# ============================================================

excluded_letters = ("A", "C", "G", "S")

task2_df = task2_df[
    ~task2_df["Category"]
    .astype(str)
    .str.upper()
    .str.startswith(excluded_letters)
].copy()

# ============================================================
# 4. FIND TOP 5 CATEGORIES
# ============================================================

top5 = (
    task2_df.groupby("Category")["Installs"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

top5_categories = top5.index.tolist()

print("Top 5 categories:")
print(top5)

# ============================================================
# 5. PREPARE CHOROPLETH DATA
# ============================================================

countries = [
    "India", "United States", "Canada", "Brazil", "Mexico",
    "United Kingdom", "France", "Germany", "Italy", "Spain",
    "Australia", "Japan", "South Korea", "Indonesia",
    "Saudi Arabia", "South Africa", "Nigeria", "Turkey",
    "Russia", "Argentina"
]

task2_top5 = task2_df[
    task2_df["Category"].isin(top5_categories)
].copy()

category_totals = (
    task2_top5.groupby("Category", as_index=False)["Installs"]
    .sum()
    .rename(columns={"Installs": "Global_Installs"})
)

map_rows = []

for _, row in category_totals.iterrows():
    for country in countries:
        map_rows.append({
            "Country": country,
            "Category": row["Category"],
            "Global_Installs": row["Global_Installs"]
        })

map_df = pd.DataFrame(map_rows)

map_df["Install_Status"] = np.where(
    map_df["Global_Installs"] > 1_000_000,
    "Above 1 Million",
    "1 Million or Less"
)

print("Choropleth data prepared.")

# ============================================================
# 6. CREATE INTERACTIVE CHOROPLETH
# ============================================================

fig_task2 = px.choropleth(
    map_df,
    locations="Country",
    locationmode="country names",
    color="Global_Installs",
    animation_frame="Category",
    hover_name="Country",
    hover_data={
        "Category": True,
        "Global_Installs": ":,",
        "Install_Status": True
    },
    color_continuous_scale="Viridis",
    title="Global Installs by Top 5 App Categories"
)

fig_task2.update_geos(
    showframe=False,
    showcoastlines=True,
    projection_type="natural earth"
)

fig_task2.update_layout(
    height=650,
    margin=dict(l=20, r=20, t=80, b=20)
)

# Include Plotly JavaScript through its CDN.
graph_html = fig_task2.to_html(
    full_html=False,
    include_plotlyjs="cdn"
)

# ============================================================
# 7. CREATE HTML WITH AUTOMATIC IST TIME CHECK
# ============================================================

dashboard_html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Task 2 - Google Play Store Analytics</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f5f5;
        }

        .dashboard {
            max-width: 1200px;
            margin: auto;
            background: white;
            padding: 25px;
            border-radius: 10px;
        }

        h1, h2 {
            text-align: center;
        }

        .info {
            padding: 15px;
            margin-bottom: 20px;
            background: #eeeeee;
            border-radius: 8px;
        }

        #unavailable {
            display: none;
            text-align: center;
            padding: 100px 15px;
        }

        #chart-container {
            display: none;
        }

        #time-status {
            text-align: center;
            font-weight: bold;
        }
    </style>
</head>

<body>

<div class="dashboard">

    <h1>Google Play Store Analytics</h1>

    <div id="unavailable">
        <h1>⏰ Task 2 is currently unavailable</h1>
        <h2>Task 2 is displayed only between 6 PM and 8 PM IST.</h2>
        <p>Please check again during the allowed time.</p>
        <p id="next-check"></p>
    </div>

    <div id="chart-container">

        <div class="info">
            <h2>Global Installs by Top 5 App Categories</h2>

            <p>
                Interactive Choropleth map showing installs
                for the top 5 eligible app categories.
            </p>

            <p>Categories beginning with A, C, G and S are excluded.</p>

            <p>Categories with installs above 1 million are highlighted.</p>

            <p><strong>Available Time:</strong> 6:00 PM - 8:00 PM IST</p>

            <p id="time-status"></p>
        </div>

        <!--PLOTLY_CHART-->

    </div>

</div>

<script>
    function checkTask2Availability() {

        // Always get the current time in Indian Standard Time.
        const parts = new Intl.DateTimeFormat("en-GB", {
            timeZone: "Asia/Kolkata",
            hour: "2-digit",
            minute: "2-digit",
            second: "2-digit",
            hourCycle: "h23"
        }).formatToParts(new Date());

        const getPart = (type) =>
            Number(parts.find(part => part.type === type).value);

        const hour = getPart("hour");
        const minute = getPart("minute");

        const currentMinutes = hour * 60 + minute;

        // 6:00 PM inclusive; 8:00 PM exclusive.
        const isActive =
            currentMinutes >= 18 * 60 &&
            currentMinutes < 20 * 60;

        const chartContainer =
            document.getElementById("chart-container");

        const unavailable =
            document.getElementById("unavailable");

        if (isActive) {

            chartContainer.style.display = "block";
            unavailable.style.display = "none";

            document.getElementById("time-status").textContent =
                "✅ It is ACTIVE Now (IST)";

            // Resize Plotly after making the chart visible.
            const chart = document.querySelector(
                "#chart-container .plotly-graph-div"
            );

            if (chart && window.Plotly) {
                window.Plotly.Plots.resize(chart);
            }

        } else {

            chartContainer.style.display = "none";
            unavailable.style.display = "block";

            document.getElementById("next-check").textContent =
                "The page will check again automatically.";
        }
    }

    // Check immediately when the page opens.
    checkTask2Availability();

    // Check again every 15 seconds.
    setInterval(checkTask2Availability, 15000);

    // Check again when the visitor returns to the browser tab.
    document.addEventListener(
        "visibilitychange",
        function () {
            if (!document.hidden) {
                checkTask2Availability();
            }
        }
    );
</script>

</body>
</html>
"""

# Insert the generated Plotly graph into the HTML.
dashboard_html = dashboard_html.replace(
    "<!--PLOTLY_CHART-->",
    graph_html
)

# ============================================================
# 8. SAVE AND OPEN DASHBOARD
# ============================================================

dashboard_path = os.path.abspath("dashboard_task2.html")

with open(dashboard_path, "w", encoding="utf-8") as file:
    file.write(dashboard_html)

print("✅ Task 2 dashboard created successfully.")
print("📁 Saved at:", dashboard_path)
print("🕕 Automatic availability: 6 PM to 8 PM IST")

webbrowser.open("file://" + dashboard_path)

Dataset loaded: (10841, 13)
Top 5 categories:
Category
PRODUCTIVITY          14176091369
TOOLS                 11452771915
FAMILY                10258263505
PHOTOGRAPHY           10088247655
NEWS_AND_MAGAZINES     7496317760
Name: Installs, dtype: int64
Choropleth data prepared.
✅ Task 2 dashboard created successfully.
📁 Saved at: c:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task2.html
🕕 Automatic availability: 6 PM to 8 PM IST


True